# 04 — Interpretación geográfica de los huecos

Localizamos sobre el mapa de CDMX las features H1 más persistentes
("huecos") de cada nivel. Cada hueco se representa por el **centroide del
ciclo** asociado a la 1-cocadena y un **radio aproximado** dado por la
persistencia (death − birth) dividido entre 2.


In [ ]:
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
import folium
from pyproj import Transformer

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
TDA_DIR = ROOT / 'data' / 'processed' / 'tda_results'

# UTM 14N -> lat/lon
inv = Transformer.from_crs(32614, 4326, always_xy=True)

def utm_to_latlon(x, y):
    lon, lat = inv.transform(x, y)
    return lat, lon


In [ ]:
def hole_centroids(r, top_k=5):
    '''Para cada feature H1 en el top-k por persistencia, recupera el
    centroide del cociclo (vértices involucrados) y un radio aproximado.'''
    dgm1 = r['dgms'][1]
    cocycles = r['cocycles'][1]  # lista de arrays (i, j, val)
    X = r['X']
    if len(dgm1) == 0:
        return []
    pers = dgm1[:,1] - dgm1[:,0]
    order = np.argsort(-pers)[:top_k]
    out = []
    for idx in order:
        b, d = dgm1[idx]
        coc = cocycles[idx]
        # vértices que participan
        verts = np.unique(coc[:, :2].ravel()) if len(coc) else np.array([], dtype=int)
        if len(verts) == 0:
            continue
        centroid = X[verts].mean(axis=0)
        out.append({
            'birth': float(b), 'death': float(d),
            'pers': float(d - b),
            'centroid_xy': centroid,
            'n_verts': len(verts),
        })
    return out


In [ ]:
# Mapa con los huecos top de cada nivel
LEVEL_COLOR = {'preescolar':'green','primaria':'blue','secundaria':'orange',
               'media_superior':'red','media_tecnica':'purple','todas':'black'}

m = folium.Map(location=[19.4326, -99.1332], zoom_start=11, tiles='cartodbpositron')
for pkl in sorted(TDA_DIR.glob('*.pkl')):
    name = pkl.stem
    if name == 'todas': continue
    with open(pkl, 'rb') as f:
        r = pickle.load(f)
    color = LEVEL_COLOR.get(name, 'gray')
    layer = folium.FeatureGroup(name=f'huecos — {name}')
    for h in hole_centroids(r, top_k=5):
        lat, lon = utm_to_latlon(*h['centroid_xy'])
        folium.Circle(
            [lat, lon], radius=h['pers']/2,
            color=color, fill=True, fill_opacity=0.15,
            popup=(f"<b>{name}</b><br>birth: {h['birth']:.0f} m<br>"
                   f"death: {h['death']:.0f} m<br>persistencia: {h['pers']:.0f} m"),
        ).add_to(layer)
        folium.CircleMarker([lat, lon], radius=4, color=color, fill=True).add_to(layer)
    layer.add_to(m)
folium.LayerControl().add_to(m)
m


In [ ]:
# Resumen tabular
rows = []
for pkl in sorted(TDA_DIR.glob('*.pkl')):
    name = pkl.stem
    with open(pkl, 'rb') as f:
        r = pickle.load(f)
    for h in hole_centroids(r, top_k=3):
        lat, lon = utm_to_latlon(*h['centroid_xy'])
        rows.append({'nivel': name, 'lat': lat, 'lon': lon,
                     'birth_m': h['birth'], 'death_m': h['death'],
                     'persistencia_m': h['pers'], 'n_verts': h['n_verts']})
pd.DataFrame(rows).sort_values('persistencia_m', ascending=False)


## Comparación con clustering clásico

DBSCAN agrupa puntos densos pero **ignora los huecos**. TDA es complementaria:
señala dónde hay déficit de cobertura aunque haya escuelas alrededor.


In [ ]:
from sklearn.cluster import DBSCAN
df = pd.read_parquet(ROOT / 'data' / 'processed' / 'escuelas_cdmx.parquet')
sub = df[df['nivel']=='primaria']
db = DBSCAN(eps=500, min_samples=5).fit(sub[['x_utm','y_utm']].values)
sub = sub.assign(cluster=db.labels_)
print('clusters DBSCAN:', sub['cluster'].nunique() - (1 if -1 in sub['cluster'].unique() else 0),
      '| ruido:', (sub['cluster']==-1).sum())


**Interpretación final.** Mientras DBSCAN cuenta densidades, TDA
identifica las regiones cerradas-por-escuelas pero vacías por dentro — las
zonas donde una política pública de planeación urbana debería considerar
abrir o reasignar planteles.
